In [ ]:
# =========================
# 0) install the package / H100 setting
# =========================
!pip -q install -U transformers accelerate datasets peft trl bitsandbytes sentencepiece huggingface_hub scikit-learn
from datasets import load_dataset
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
# =========================
# 1) Hugging Face set up
# =========================
login("")


In [ ]:
!pip -q install -U pandas==2.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 163.9 MB/s eta 0:00:00


In [ ]:
# =========================
# 2) Read the data
# =========================

data_path = "/content/train.jsonl"
ds = load_dataset("json", data_files=data_path)["train"]

split = ds.train_test_split(test_size=0.05, seed=42)
train_raw = split["train"]
valid_raw = split["test"]

def build_text(example):
    instruction = example.get("instruction", "").strip()
    user_input = example.get("input", "").strip()
    output = example.get("output", "").strip()

    # Mistral Instruct 建議用 [INST] 格式
    text = (
        f"{tokenizer.bos_token}"
        f"[INST] {instruction}\n{user_input} [/INST] {output}{tokenizer.eos_token}"
    )
    return {"text": text}

train_ds = train_raw.map(build_text, remove_columns=train_raw.column_names)
valid_ds = valid_raw.map(build_text, remove_columns=valid_raw.column_names)

# =========================
# 3) import tokenizer / 4-bit Mistral
# =========================

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

# =========================
# 4) LoRA
# =========================

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# =========================
# 5) Model training
# =========================

training_args = SFTConfig(
    output_dir="/content/mistral_lora_out",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
    max_length=128,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
)

torch.cuda.empty_cache()
trainer.train()

# =========================
# 6) save LoRA adapter / early stopped!!
# =========================
save_dir = "/content/mistral_lora_adapter"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved to:", save_dir)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

trainable params: 20,971,520 || all params: 7,262,703,616 || trainable%: 0.2888


Adding EOS to train dataset:   0%|          | 0/14250 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/14250 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
100,1.511803,1.416948
200,1.442527,1.394367


Step,Training Loss,Validation Loss
100,1.511803,1.416948
200,1.442527,1.394367
300,1.531635,1.403451
400,1.455527,1.390182
500,1.433951,1.385854
600,1.568168,1.392436
700,1.375632,1.383916
800,1.464086,1.383160
900,1.551122,1.389186
1000,1.629097,1.388726


KeyboardInterrupt: 

In [ ]:
# =========================
# Mistral-7B + LoRA evaluation
# =========================

import re
import gc
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# ====== directory ======
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
checkpoint_path = "/content/mistral_lora_out/checkpoint-1300"

# ====== tokenizer ======
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ====== 4-bit base model ======
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, checkpoint_path)
model.eval()

# ====== label cleanse ======
def normalize_label(text: str) -> str:
    text = (text or "").strip().lower()
    text = re.sub(r"[\n\r\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9_ \-]", "", text)
    return text.strip()

# ====== prompt ======
def build_eval_prompt(example):
    instruction = example.get("instruction", "").strip()
    user_input = example.get("input", "").strip()
    prompt = f"{tokenizer.bos_token}[INST] {instruction}\n{user_input} [/INST]"
    return prompt

# ====== evaluation======
def predict_label(example):
    prompt = build_eval_prompt(example)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    pred = decoded.split("[/INST]")[-1].strip().split("\n")[0]
    return normalize_label(pred)

# ====== get eval set ======
eval_data = valid_raw

# ====== run the evaluation ======
y_true = []
y_pred = []

for ex in eval_data:
    true_label = normalize_label(ex.get("output", ""))
    pred_label = predict_label(ex)

    y_true.append(true_label)
    y_pred.append(pred_label)

labels = sorted(list(set(y_true) | set(y_pred)))

acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
report = classification_report(y_true, y_pred, labels=labels, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=labels)

print("Accuracy:", acc)
print("Macro F1:", f1_macro)
print("\nClassification Report:\n")
print(report)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{x}" for x in labels],
    columns=[f"pred_{x}" for x in labels],
)

display(cm_df)

# ====== release the memory ======
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for ope

Accuracy: 0.6106666666666667
Macro F1: 0.40403124191613154

Classification Report:

              precision    recall  f1-score   support

       anger       0.75      0.10      0.18        60
     disgust       1.00      0.31      0.48        16
        fear       0.29      0.74      0.42        35
         joy       0.75      0.32      0.45       186
     neutral       0.66      0.92      0.77       366
     sadness       0.43      0.38      0.41        60
    surprise       1.00      0.07      0.14        27

    accuracy                           0.61       750
   macro avg       0.70      0.41      0.40       750
weighted avg       0.67      0.61      0.57       750



,pred_anger,pred_disgust,pred_fear,pred_joy,pred_neutral,pred_sadness,pred_surprise
true_anger,6,0,9,2,35,8,0
true_disgust,0,5,4,0,7,0,0
true_fear,0,0,26,1,6,2,0
true_joy,1,0,17,59,99,10,0
true_neutral,0,0,14,6,337,9,0
true_sadness,1,0,13,6,17,23,0
true_surprise,0,0,6,5,13,1,2


In [17]:
!du -sh /content/mistral_lora_out/

170M	/content/mistral_lora_out/


In [18]:
import shutil
shutil.make_archive('mistral_lora_out', 'zip', '/content/mistral_lora_out/')

'/content/mistral_lora_out.zip'

In [19]:
from google.colab import files
files.download('mistral_lora_out.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>